# 05 — How much of the tour-level gap does each cause explain?

Notebook 04 measured the *matrix* gap. But our results diverge at the **tour**
level, where three more mechanisms pile on top: the model's service-time
assumption, its time-window handling (including two upstream bugs), and solver
noise. This notebook quantifies each one, using two instruments:

1. **Fixed-sequence re-evaluation** — re-cost Swiss Post's *own* stop sequences
   under our assumptions. No optimization involved, so differences are *pure*
   assumption effects.
2. **PyVRP re-solves** — re-optimize the same days under different matrices, to
   see when the *routes themselves* change (the optimizer reacting to the
   matrix, cf. notebook 01 §5).

**Inputs**: `outputs/benchmark_tours.json` (nb 03), `outputs/summary_od_comparison.json` (nb 04).

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sppdptw import config, evaluate, extraction, io, osm

instance, sol, M_SP, index_mapping = io.load_all()
nodes = extraction.build_nodes_df(instance, index_mapping)
visit_times = extraction.get_visit_times(nodes)

with open(config.OUTPUTS_DIR / "benchmark_tours.json") as f:
    benchmark = json.load(f)
tours = pd.DataFrame(benchmark["tours"])
with open(config.OUTPUTS_DIR / "summary_od_comparison.json") as f:
    od = json.load(f)

k_hat, c_hat = od["affine_k"], od["affine_c_s"]
print(f"{len(tours)} benchmark tours | OD calibration: "
      f"SP ≈ {k_hat:.3f}·OSM + {c_hat:.0f}s  [{od['source']}]")

# OSM matrix (same source logic as notebook 04)
if osm.MATRIX_CACHE.exists():
    M_OSM, _ = osm.build_osm_matrix(nodes)
    osm_source = "OSMnx build"
else:
    M_OSM, _ = osm.demo_matrix(M_SP, nodes)
    osm_source = "DEMO"
M_OSM_f = np.nan_to_num(M_OSM, nan=float(np.nanmax(M_OSM) * 2))
M_OSM_cal = k_hat * M_OSM_f + c_hat  # affine-calibrated OSM
np.fill_diagonal(M_OSM_cal, 0.0)

## 1. Fixed-sequence re-evaluation: assumption effects, isolated

Take every Swiss Post tour *exactly as driven* (same stops, same order) and
re-cost it under different (matrix, service-time) assumptions. Any change is
attributable to the assumption alone.

| scenario | matrix | service time | isolates |
|---|---|---|---|
| `baseline` | Swiss Post | per-location `visit_time` | (must reproduce the benchmark) |
| `service_5min` | Swiss Post | uniform 5 min | the model's service assumption |
| `osm` | OSM free-flow | `visit_time` | the raw matrix gap |
| `osm_calibrated` | k̂·OSM + ĉ | `visit_time` | matrix gap left AFTER calibration |

In [ ]:
scenarios = {
    "baseline": (M_SP, visit_times),
    "service_5min": (M_SP, 300.0),
    "osm": (M_OSM_f, visit_times),
    "osm_calibrated": (M_OSM_cal, visit_times),
}
evals = {
    name: evaluate.evaluate_tours(tours, m, vt, label=name)
    for name, (m, vt) in scenarios.items()
}

# validation: baseline must reproduce the solution's own drive times exactly
assert np.allclose(evals["baseline"].drive_min, tours.drive_min, atol=0.01), \
    "baseline re-evaluation failed to reproduce the benchmark"
print("baseline reproduces every benchmark tour exactly ✓")

rows = []
for name, ev in evals.items():
    rows.append({
        "scenario": name,
        "drive_min_total": ev.drive_min.sum(),
        "service_min_total": ev.service_min.sum(),
        "total_min": ev.total_min.sum(),
        "vs_benchmark_pct": 100 * (
            ev.total_min.sum() / evals["baseline"].total_min.sum() - 1
        ),
    })
table = pd.DataFrame(rows).set_index("scenario")
print(table.round(1).to_string())

In [ ]:
base_drive = evals["baseline"].drive_min
fig, ax = plt.subplots(figsize=(9, 4.5))
for name, color in [("osm", "tab:red"), ("osm_calibrated", "tab:green")]:
    delta = evals[name].drive_min - base_drive
    ax.scatter(base_drive, delta, s=14, alpha=0.6, c=color, label=name)
ax.axhline(0, c="k", lw=0.8)
ax.set_xlabel("benchmark tour drive (min)")
ax.set_ylabel("drive difference vs benchmark (min)")
ax.set_title(f"Per-tour drive error of the OSM matrix, before/after calibration "
             f"[{osm_source}]")
ax.legend(); plt.tight_layout(); plt.show()

raw_err = (evals["osm"].drive_min / base_drive - 1)
cal_err = (evals["osm_calibrated"].drive_min / base_drive - 1)
print(f"raw OSM        : tours drive {raw_err.mean():+.1%} ± {raw_err.std():.1%} vs benchmark")
print(f"calibrated OSM : tours drive {cal_err.mean():+.1%} ± {cal_err.std():.1%} vs benchmark")

## 2. The service-time assumption

The `service_5min` row above shows what the uniform assumption does to *tour
totals*. Per stop, the error depends on which locations a tour visits:

In [ ]:
sv_err = evals["service_5min"].service_min - evals["baseline"].service_min
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sv_err, bins=30)
ax.axvline(0, c="k", lw=0.8)
ax.set_xlabel("service-time error per tour (min), uniform 5min − real visit_time")
ax.set_title("What 'service = 5 min per stop' does to each tour")
plt.tight_layout(); plt.show()
print(f"per-tour service error: mean {sv_err.mean():+.1f} min, "
      f"range [{sv_err.min():.0f}, {sv_err.max():.0f}] min")

## 3. The time-window handling (two upstream bugs, quantified on data)

**(a) zero-width drop-off windows.** The upstream extraction turns every
transport drop-off window `(start, end)` into `(start, start)`. The data's
windows are all 30 min wide; the bug then re-widens them only by
`(n_deliveries−1)·5 min` at shared stops. Net effect on the feasible space:

In [ ]:
ship_ok = extraction.build_shipments_df(instance, nodes)
ship_bug = extraction.build_shipments_df(instance, nodes, buggy_dropoff_tw=True)

tw = ship_ok[ship_ok.kind == "transport"].copy()
tw["real_width"] = tw.dropoff_tw.apply(lambda t: t[1] - t[0])
# the make_the_TW_feasible patch re-widens PER-DAY (location, time) groups
grp = tw.groupby(
    ["date", tw.to_idx, tw.dropoff_tw.apply(lambda t: t[0])]
).size()
patched_width = (grp - 1) * 5 + 0.1
print(f"real drop-off windows      : all {tw.real_width.unique()} min wide")
print(f"after bug + feasibility fix: median {patched_width.median():.1f} min, "
      f"max {patched_width.max():.1f} min")
print(f"=> the model must hit a ~{patched_width.median():.1f}-min target instead "
      f"of a 30-min window, for {len(tw)} shipments")

A vehicle forced to arrive in a ≈0-min window either waits (inflating tour
time), detours (inflating drive time), or **skips the shipment** (the `y`
variables) — all three push the model's results away from Swiss Post's, and
none of them is the matrix's fault.

**(b) `apply_TW_per_date` discards the pickup fix** — its second call passes
the *original* day dataframe instead of the output of the first call
(`pdptw/helpers.py`), so pickup windows keep any infeasible `(t, t)` forms.

**(c) the 6-hour driver limit.** The model fixes the tour span to
07:00–17:00 (10 h) with no break; the instance's driver template allows at
most **6 h** plus a 30-min break, and the instance's cost structure penalizes
overtime (`tour_overtime_cost`) — a *soft* limit Swiss Post's solver trades
off, while the pdptw model ignores it entirely:

In [ ]:
print(f"benchmark tour durations: max {tours.total_min.max():.0f} min, "
      f"{(tours.total_min > 360).sum()} of {len(tours)} exceed 360 min")
print("(overtime happens, but it is *penalized* in Swiss Post's objective — "
      "a model with a free 10-hour span faces no such pressure)")

## 4. Re-optimization: does the matrix change the *routes*?

Fixed sequences isolate cost effects; but an optimizer *reacts* to the matrix.
We re-solve four representative days (the report's date set) as a VRPTW over
the hub shipments with PyVRP, under three matrices, and measure:

- the **objective** (drive seconds) per matrix — should scale ≈ k̂ if the gap
  is uniform;
- the **route overlap** (Jaccard similarity of driven arcs) between the SP-matrix
  solution and the OSM-matrix solution, against a **seed-noise baseline** —
  if (SP vs OSM) overlap is inside the noise band, the matrix gap does not
  change routing decisions; far below it, the gap is structurally distorting.

In [ ]:
import pyvrp
import pyvrp.stop

DATES = ["2023-07-10", "2023-07-21", "2023-09-05", "2023-08-28"]
HUB_IDX = int(nodes[nodes.is_hub & nodes.in_areas.apply(
    lambda a: config.AREA_ID in a)]["index"].iloc[0])


def solve_day(date: str, matrix: np.ndarray, seed: int = 0,
              runtime_s: int = 3) -> dict:
    """VRPTW over the day's hub shipments; returns objective + driven arcs."""
    day = ship_ok[(ship_ok.date == date) & (ship_ok.kind == "hub")]
    clients_idx = []  # position -> matrix index
    m = pyvrp.Model()
    depot = m.add_depot(x=0, y=0)
    m.add_vehicle_type(
        num_available=3,
        capacity=int(1_039_000),
        tw_early=420 * 60, tw_late=1020 * 60,
        shift_duration=6 * 3600,  # the instance's 6-hour driver limit
        start_depot=depot, end_depot=depot,
    )
    for _, s in day.iterrows():
        loc = int(s.to_idx if s.to_idx != HUB_IDX else s.from_idx)
        is_delivery = s.to_idx != HUB_IDX
        clients_idx.append(loc)
        m.add_client(
            x=len(clients_idx), y=0,
            delivery=int(s.weight) if is_delivery else 0,
            pickup=0 if is_delivery else int(s.weight),
            service_duration=int(visit_times.get(loc, 0)),
            tw_early=int(s.dropoff_tw[0] * 60), tw_late=int(s.dropoff_tw[1] * 60),
        )
    locs = [HUB_IDX] + clients_idx
    for a, fa in enumerate(m.locations):
        for b, fb in enumerate(m.locations):
            if a != b:
                t = int(round(matrix[locs[a], locs[b]]))
                m.add_edge(fa, fb, distance=t, duration=t)
    res = m.solve(stop=pyvrp.stop.MaxRuntime(runtime_s), seed=seed, display=False)
    arcs = set()
    for route in res.best.routes():
        seq = [0] + list(route.visits()) + [0]
        arcs |= {(locs[a], locs[b]) for a, b in zip(seq[:-1], seq[1:])}
    return {"objective": res.cost(), "arcs": arcs,
            "feasible": res.best.is_feasible()}


def jaccard(a: set, b: set) -> float:
    return len(a & b) / len(a | b) if a | b else 1.0


records = []
for date in DATES:
    sp0 = solve_day(date, M_SP, seed=0)
    noise = [jaccard(sp0["arcs"], solve_day(date, M_SP, seed=s)["arcs"])
             for s in (1, 2, 3)]
    osm_sol = solve_day(date, M_OSM_f, seed=0)
    cal_sol = solve_day(date, M_OSM_cal, seed=0)
    records.append({
        "date": date,
        "sp_obj_min": sp0["objective"] / 60,
        "osm_obj_min": osm_sol["objective"] / 60,
        "cal_obj_min": cal_sol["objective"] / 60,
        "overlap_noise_baseline": np.mean(noise),
        "overlap_sp_vs_osm": jaccard(sp0["arcs"], osm_sol["arcs"]),
        "overlap_sp_vs_cal": jaccard(sp0["arcs"], cal_sol["arcs"]),
    })
solves = pd.DataFrame(records)
print(solves.round(3).to_string(index=False))
print(f"\nobjective ratio sp/osm: {(solves.sp_obj_min / solves.osm_obj_min).mean():.3f} "
      f"(matrix k̂ = {k_hat:.3f})")

Reading this table:

- `overlap_noise_baseline` is how much the routes change when *only the random
  seed* changes — the intrinsic wobble of a heuristic solver. Any overlap value
  above/near this baseline means "indistinguishable from noise".
- If `overlap_sp_vs_osm` ≪ baseline, the OSM matrix *reorders* tours — evidence
  that the matrix gap is structurally distorting the optimization (short-trip
  overhead and congestion pockets change which neighbor is 'closest').
- The objective ratio should land near k̂ when the gap is mostly uniform.

## 5. Putting the pieces together

In [ ]:
gap_matrix = float(raw_err.mean())
gap_matrix_after_cal = float(cal_err.mean())
gap_service = float(
    (evals["service_5min"].total_min.sum() / evals["baseline"].total_min.sum()) - 1
)
summary = {
    "osm_source": osm_source,
    "matrix_gap_drive_pct": gap_matrix * 100,
    "matrix_gap_after_calibration_pct": gap_matrix_after_cal * 100,
    "calibration": {"k": k_hat, "c_s": c_hat, "shape": od["gap_shape"]},
    "service_assumption_total_pct": gap_service * 100,
    "tw_bug_shipments_affected": int(len(tw)),
    "tw_real_width_min": 30,
    "solver_noise_route_overlap": float(solves.overlap_noise_baseline.mean()),
    "sp_vs_osm_route_overlap": float(solves.overlap_sp_vs_osm.mean()),
    "sp_vs_calibrated_route_overlap": float(solves.overlap_sp_vs_cal.mean()),
}
with open(config.OUTPUTS_DIR / "summary_faithfulness.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

### Verdict template

| Divergence source | Measured here | Fix |
|---|---|---|
| **Matrix gap** | raw OSM underestimates drive by the % in §1; affine calibration removes most of it | use k̂·OSM + ĉ (notebook 04), or a routing engine with congestion profiles |
| **Service times** | §2: uniform 5 min mis-costs tours by tens of minutes either way | use the instance's per-location `visit_time` |
| **Drop-off TW bug** | §3: 128 shipments constrained to ~0-min instead of 30-min windows | fix `(start, start)` → `(start, end)` in the extraction |
| **Pickup TW fix discarded** | §3(b) | pass `fixed_day` to the second `make_the_TW_feasible` call |
| **Driver limits** | §3(c): 6 h + break vs the model's fixed 10 h span | add `max_duration`, break, and free end time to the MILP |
| **Solver noise** | §4 baseline overlap | report seeds/gaps; compare against the noise band, not point values |

**The bottom line for the original question** — *"is it a static shift?"* —
is notebook 04's verdict (printed above in §5's summary): on this data the
matrix gap is *near*-uniform (affine), so the OD methodology is fundamentally
sound **but** a tour-level match with Swiss Post additionally requires fixing
the service-time assumption and the time-window extraction, which have effects
of the same order of magnitude as the matrix gap itself.